# Étude de Cas: FIFA World Cup 2026 pipeline development

This notebook reads the Bronze Delta Table, expands the nested JSON structure, flattens the fields, and writes a clean Silver Delta Table ready for analytics

In [0]:
# Step 0: Load Bronze Delta Table into a Spark DataFrame
bronze_df = spark.read.table("workspace.default.bronze_fixtures")
display(bronze_df.limit(5))

fixture,goals,league,score,teams
"List(2023-03-23T23:30:00+00:00, 1012323, List(1679614200, 1679617800), C. Ferreyra, List(90, null, Match Finished, FT), 1679614200, UTC, List(Capital Federal, Ciudad de Buenos Aires, 19570, Estadio Mâs Monumental))","List(0, 2)","List(World, null, 10, https://media.api-sports.io/football/leagues/10.png, Friendlies, Friendlies 1, 2023, false)","List(List(null, null), List(0, 2), List(0, 0), List(null, null))","List(List(11, https://media.api-sports.io/football/teams/11.png, Panama, false), List(26, https://media.api-sports.io/football/teams/26.png, Argentina, true))"
"List(2023-03-28T23:30:00+00:00, 1012339, List(1680046200, 1680049800), G. Tejera, List(90, null, Match Finished, FT), 1680046200, UTC, List(Santiago del Estero, Prov. de Santiago del Estero, null, Estadio Único Madre de Ciudades))","List(0, 7)","List(World, null, 10, https://media.api-sports.io/football/leagues/10.png, Friendlies, Friendlies 1, 2023, false)","List(List(null, null), List(0, 7), List(0, 5), List(null, null))","List(List(5530, https://media.api-sports.io/football/teams/5530.png, Curaçao, false), List(26, https://media.api-sports.io/football/teams/26.png, Argentina, true))"
"List(2023-06-15T12:00:00+00:00, 1028642, List(1686830400, 1686834000), Ning Ma, China, List(90, null, Match Finished, FT), 1686830400, UTC, List(Beijing, 348, Workers' Stadium))","List(0, 2)","List(World, null, 10, https://media.api-sports.io/football/leagues/10.png, Friendlies, Friendlies 1, 2023, false)","List(List(null, null), List(0, 2), List(0, 1), List(null, null))","List(List(20, https://media.api-sports.io/football/teams/20.png, Australia, false), List(26, https://media.api-sports.io/football/teams/26.png, Argentina, true))"
"List(2023-06-19T12:30:00+00:00, 1028654, List(1687177800, 1687181400), Usaid Jamal, List(90, null, Match Finished, FT), 1687177800, UTC, List(Jakarta, 3892, Stadion Utama Gelora Bung Karno))","List(2, 0)","List(World, null, 10, https://media.api-sports.io/football/leagues/10.png, Friendlies, Friendlies 1, 2023, false)","List(List(null, null), List(2, 0), List(1, 0), List(null, null))","List(List(26, https://media.api-sports.io/football/teams/26.png, Argentina, true), List(1571, https://media.api-sports.io/football/teams/1571.png, Indonesia, false))"
"List(2024-06-26T01:00:00+00:00, 1146307, List(1719363600, 1719367200), A. Matonte, List(90, null, Match Finished, FT), 1719363600, UTC, List(East Rutherford, New Jersey, null, MetLife Stadium))","List(1, 0)","List(World, null, 9, https://media.api-sports.io/football/leagues/9.png, Copa America, Group Stage - 2, 2024, true)","List(List(null, null), List(1, 0), List(0, 0), List(null, null))","List(List(26, https://media.api-sports.io/football/teams/26.png, Argentina, true), List(2383, https://media.api-sports.io/football/teams/2383.png, Chile, false))"


In [0]:
# Step 1: Filter Out Corrupt Records
bronze_df.filter(col("_corrupt_record").isNotNull()).count()
#It failed, so the files were uploaded correctly

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-4886221969004087>, line 2
      1 # Step 1: Filter Out Corrupt Records
----> 2 bronze_df.filter(col("_corrupt_record").isNotNull()).count()
      3 #It failed, so the files were uploaded correctly

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/dataframe.py:318, in DataFrame.count(self)
    315 def count(self) -> int:
    316     table, _ = self.agg(
    317         F._invoke_function("count", F.lit(1))
--> 318     )._to_table()  # type: ignore[operator]
    319     return table[0][0].as_py()

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/dataframe.py:1930, in DataFrame._to_table(self)
   1928 def _to_table(self) -> Tuple["pa.Table", Optional[StructType]]:
   1929     query = self._plan.to_proto(self._session.client)
-> 1930     table, schema, self._execution_info = self


Typically, APIs return a JSON structure like:

```json
{
  "get": "...",
  "parameters": {...},
  "response": [ {...}, {...} ]
}
```

In which we need to explode the `response` array to form a tabular object using:

```python3
from pyspark.sql.functions import col, explode

expanded = bronze_clean.select(
    explode(col("response.col1")).alias("col1")
    ...
    explode(col("response.coln")).alias("coln")
)
```

In [0]:
# Step 2: Flatten the Struct

from pyspark.sql.functions import col, explode

silver_df = bronze_df.select(
        # fixture.*
        col("fixture.date").alias("fixture_date"),
        col("fixture.id").alias("fixture_id"),
        col("fixture.periods.first").alias("period_first"),
        col("fixture.periods.second").alias("period_second"),
        col("fixture.referee").alias("referee"),
        col("fixture.status.elapsed").alias("status_elapsed"),
        col("fixture.status.extra").alias("status_extra"),
        col("fixture.status.long").alias("status_long"),
        col("fixture.status.short").alias("status_short"),
        col("fixture.timestamp").alias("timestamp"),
        col("fixture.timezone").alias("timezone"),
        col("fixture.venue.city").alias("venue_city"),
        col("fixture.venue.id").alias("venue_id"),
        col("fixture.venue.name").alias("venue_name"),
    # goals.*
        col("goals.home").alias("goals_home"),
        col("goals.away").alias("goals_away"),
    # league.*
        col("league.country").alias("league_country"),
        col("league.flag").alias("league_flag"),
        col("league.id").alias("league_id"),
        col("league.logo").alias("league_logo"),
        col("league.name").alias("league_name"),
        col("league.round").alias("league_round"),
        col("league.season").alias("league_season"),
        col("league.standings").alias("league_standings"),
    # score.*
        col("score.extratime.home").alias("extratime_home"),
        col("score.extratime.away").alias("extratime_away"),
        col("score.fulltime.home").alias("fulltime_home"),
        col("score.fulltime.away").alias("fulltime_away"),
        col("score.halftime.home").alias("halftime_home"),
        col("score.halftime.away").alias("halftime_away"),
        col("score.penalty.home").alias("penalty_home"),
        col("score.penalty.away").alias("penalty_away"),
    # teams.*
        col("teams.home.id").alias("home_team_id"),
        col("teams.home.name").alias("home_team_name"),
        col("teams.home.logo").alias("home_team_logo"),
        col("teams.home.winner").alias("home_team_winner"),
    col("teams.away.id").alias("away_team_id"),
        col("teams.away.name").alias("away_team_name"),
        col("teams.away.logo").alias("away_team_logo"),
        col("teams.away.winner").alias("away_team_winner")
    )

#display(silver_df.limit(5))

In [0]:
display(silver_df.limit(5))

fixture_date,fixture_id,period_first,period_second,referee,status_elapsed,status_extra,status_long,status_short,timestamp,timezone,venue_city,venue_id,venue_name,goals_home,goals_away,league_country,league_flag,league_id,league_logo,league_name,league_round,league_season,league_standings,extratime_home,extratime_away,fulltime_home,fulltime_away,halftime_home,halftime_away,penalty_home,penalty_away,home_team_id,home_team_name,home_team_logo,home_team_winner,away_team_id,away_team_name,away_team_logo,away_team_winner
2023-03-23T23:30:00+00:00,1012323,1679614200,1679617800,C. Ferreyra,90,null,Match Finished,FT,1679614200,UTC,"Capital Federal, Ciudad de Buenos Aires",19570,Estadio Mâs Monumental,2,0,World,null,10,https://media.api-sports.io/football/leagues/10.png,Friendlies,Friendlies 1,2023,false,null,null,2,0,0,0,null,null,26,Argentina,https://media.api-sports.io/football/teams/26.png,true,11,Panama,https://media.api-sports.io/football/teams/11.png,false
2023-03-28T23:30:00+00:00,1012339,1680046200,1680049800,G. Tejera,90,null,Match Finished,FT,1680046200,UTC,"Santiago del Estero, Prov. de Santiago del Estero",null,Estadio Único Madre de Ciudades,7,0,World,null,10,https://media.api-sports.io/football/leagues/10.png,Friendlies,Friendlies 1,2023,false,null,null,7,0,5,0,null,null,26,Argentina,https://media.api-sports.io/football/teams/26.png,true,5530,Curaçao,https://media.api-sports.io/football/teams/5530.png,false
2023-06-15T12:00:00+00:00,1028642,1686830400,1686834000,"Ning Ma, China",90,null,Match Finished,FT,1686830400,UTC,Beijing,348,Workers' Stadium,2,0,World,null,10,https://media.api-sports.io/football/leagues/10.png,Friendlies,Friendlies 1,2023,false,null,null,2,0,1,0,null,null,26,Argentina,https://media.api-sports.io/football/teams/26.png,true,20,Australia,https://media.api-sports.io/football/teams/20.png,false
2023-06-19T12:30:00+00:00,1028654,1687177800,1687181400,Usaid Jamal,90,null,Match Finished,FT,1687177800,UTC,Jakarta,3892,Stadion Utama Gelora Bung Karno,0,2,World,null,10,https://media.api-sports.io/football/leagues/10.png,Friendlies,Friendlies 1,2023,false,null,null,0,2,0,1,null,null,1571,Indonesia,https://media.api-sports.io/football/teams/1571.png,false,26,Argentina,https://media.api-sports.io/football/teams/26.png,true
2024-06-26T01:00:00+00:00,1146307,1719363600,1719367200,A. Matonte,90,null,Match Finished,FT,1719363600,UTC,"East Rutherford, New Jersey",null,MetLife Stadium,0,1,World,null,9,https://media.api-sports.io/football/leagues/9.png,Copa America,Group Stage - 2,2024,true,null,null,0,1,0,0,null,null,2383,Chile,https://media.api-sports.io/football/teams/2383.png,false,26,Argentina,https://media.api-sports.io/football/teams/26.png,true


In [0]:
%sql
-- Step 3: Create Silver Delta Table
CREATE TABLE IF NOT EXISTS workspace.default.silver_fixtures
USING DELTA;

In [0]:
# Step 4: Dedup the Data
from pyspark.sql import functions as F

silver_dedup = (
    silver_df
    .withColumn("ingest_ts", F.current_timestamp())
    .orderBy(F.col("ingest_ts").desc())
    .dropDuplicates(["fixture_id"])
)

print(f"Total={silver_df.count()}, Únicos={silver_df.distinct().count()}")

print(f"Total={silver_dedup.count()}, Únicos={silver_dedup.distinct().count()}")

Total=525, Únicos=352
Total=352, Únicos=352


In [0]:
# Alternative: Step 4: Dedup the Data with Window Functions
from pyspark.sql.window import Window
from pyspark.sql import functions as F

w = Window.partitionBy("fixture_id").orderBy(F.col("fixture_date").desc())

silver_dedup = (
    silver_df
    .withColumn("rn", F.row_number().over(w))
    .filter("rn = 1")
    .drop("rn")
)


In [0]:
# Alternative: Step 4: Use applyChangesInto but only works in DLT or Lakeflow
from pyspark.sql.functions import *

(
  silver_df
  .select("*")
  .applyChangesInto(
      target = "workspace.default.silver_fixtures",
      keys = ["fixture_id"],
      sequenceBy = col("fixture_date"),
      applyAsDeletions = False
  )
)


In [0]:
# Step 5: MERGE with Silver Delta Table and Pyspark DataFrame for incremental Silver Layer
from delta.tables import DeltaTable

deltaTable = DeltaTable.forName(spark, "workspace.default.silver_fixtures")

(
    deltaTable.alias("t")
    .merge(
        silver_dedup.alias("s"),
        "t.fixture_id = s.fixture_id"
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
%sql
-- Step 6: Validate Silver Table Creation
SELECT * FROM workspace.default.silver_fixtures

fixture_date,fixture_id,period_first,period_second,referee,status_elapsed,status_extra,status_long,status_short,timestamp,timezone,venue_city,venue_id,venue_name,goals_home,goals_away,league_country,league_flag,league_id,league_logo,league_name,league_round,league_season,league_standings,extratime_home,extratime_away,fulltime_home,fulltime_away,halftime_home,halftime_away,penalty_home,penalty_away,home_team_id,home_team_name,home_team_logo,home_team_winner,away_team_id,away_team_name,away_team_logo,away_team_winner
2023-03-23T19:45:00+00:00,980457,1679600700,1679604300,S. Jovanović,90,null,Match Finished,FT,1679600700,UTC,Napoli,11904,Stadio Diego Armando Maradona,1,2,World,null,960,https://media.api-sports.io/football/leagues/960.png,Euro Championship - Qualification,Qualifying Round - 1,2023,true,null,null,1,2,0,2,null,null,768,Italy,https://media.api-sports.io/football/teams/768.png,false,10,England,https://media.api-sports.io/football/teams/10.png,true
2023-03-26T16:00:00+00:00,980480,1679846400,1679850000,S. Gözübüyük,90,null,Match Finished,FT,1679846400,UTC,London,489,Wembley Stadium,2,0,World,null,960,https://media.api-sports.io/football/leagues/960.png,Euro Championship - Qualification,Qualifying Round - 2,2023,true,null,null,2,0,2,0,null,null,10,England,https://media.api-sports.io/football/teams/10.png,true,772,Ukraine,https://media.api-sports.io/football/teams/772.png,false
2023-06-16T18:45:00+00:00,980502,1686941100,1686944700,I. Pajač,90,null,Match Finished,FT,1686941100,UTC,Ta'Qali,2614,Ta'Qali National Stadium,0,4,World,null,960,https://media.api-sports.io/football/leagues/960.png,Euro Championship - Qualification,Qualifying Round - 3,2023,true,null,null,0,4,0,3,null,null,1112,Malta,https://media.api-sports.io/football/teams/1112.png,false,10,England,https://media.api-sports.io/football/teams/10.png,true
2023-06-19T18:45:00+00:00,980526,1687200300,1687203900,I. Kovács,90,null,Match Finished,FT,1687200300,UTC,Manchester,556,Old Trafford,7,0,World,null,960,https://media.api-sports.io/football/leagues/960.png,Euro Championship - Qualification,Qualifying Round - 4,2023,true,null,null,7,0,3,0,null,null,10,England,https://media.api-sports.io/football/teams/10.png,true,1105,FYR Macedonia,https://media.api-sports.io/football/teams/1105.png,false
2023-09-09T16:00:00+00:00,980565,1694275200,1694278800,G. Kabakov,90,null,Match Finished,FT,1694275200,UTC,Wrocław,18628,Tarczyński Arena,1,1,World,null,960,https://media.api-sports.io/football/leagues/960.png,Euro Championship - Qualification,Qualifying Round - 5,2023,true,null,null,1,1,1,1,null,null,772,Ukraine,https://media.api-sports.io/football/teams/772.png,null,10,England,https://media.api-sports.io/football/teams/10.png,null
2023-10-17T18:45:00+00:00,980633,1697568300,1697571900,C. Turpin,90,null,Match Finished,FT,1697568300,UTC,London,489,Wembley Stadium,3,1,World,null,960,https://media.api-sports.io/football/leagues/960.png,Euro Championship - Qualification,Qualifying Round - 8,2023,true,null,null,3,1,1,1,null,null,10,England,https://media.api-sports.io/football/teams/10.png,true,768,Italy,https://media.api-sports.io/football/teams/768.png,false
2023-11-17T19:45:00+00:00,980651,1700250300,1700253900,Luís Godinho,90,null,Match Finished,FT,1700250300,UTC,London,489,Wembley Stadium,2,0,World,null,960,https://media.api-sports.io/football/leagues/960.png,Euro Championship - Qualification,Qualifying Round - 9,2023,true,null,null,2,0,1,0,null,null,10,England,https://media.api-sports.io/football/teams/10.png,true,1112,Malta,https://media.api-sports.io/football/teams/1112.png,false
2023-11-20T19:45:00+00:00,980672,1700509500,1700513100,F. Glova,90,null,Match Finished,FT,1700509500,UTC,Skopje,1045,Toše Proeski Arena,1,1,World,null,960,https://media.api-sports.io/football/leagues/960.png,Euro Championship - Qualification,Qualifying Round - 10,2023,true,null,null,1,1,1,0,null,null,1105,FYR Macedonia,https://media.api-sports.io/football/teams/1105.png,null,10,England,https://media.a